# Week 08: Tools and controlled actions

**Goal:** Implement a bounded read-only tool and reason about approval for actions.

This notebook runs with the Python standard library. All records are fictional. Run cells top to bottom, then change one input and explain the observed result. Classroom stand-ins are labelled; they are not trained generative models.

## 1. Set up the case

The assistant calculates a permitted reimbursement cap; a request to update a record needs approval.

In [ ]:
from decimal import Decimal, InvalidOperation
def capped_reimbursement(amount,cap='40'):
    try: value=Decimal(str(amount))
    except InvalidOperation: raise ValueError('amount must be numeric')
    if not value.is_finite() or value<0: raise ValueError('amount must be nonnegative and finite')
    return min(value,Decimal(cap))
def mock_write(action,role,approved=False):
    if role!='manager' or not approved: return {'status':'blocked','changed':False}
    return {'status':'simulated approval','changed':False}  # no actual write
for x in ['55','10','-1','oops']:
    try: print(x,capped_reimbursement(x))
    except ValueError as e: print(x,'blocked:',e)
print(mock_write('change payroll','student',approved=True))
assert mock_write('change payroll','student',True)['changed'] is False


The mock action never writes data. Describe the missing authentication, trusted approver identity and audit log required for a real action.

## 3. Record your evidence

| Original setting | Your one change | Observed output | Explanation | Limitation |
|---|---|---|---|---|
| | | | | |

**Exit question:** A tool call chosen by a model is automatically authorized. Explain why this is wrong.

**Reading:** [Anthropic, Building effective agents](https://www.anthropic.com/engineering/building-effective-agents), [OpenAI, Agent safety](https://developers.openai.com/api/docs/guides/agent-builder-safety)

## Approval flow and a blocked request
This only changes an in-memory fictional record. A real service needs authenticated identities and approval bound to the exact action.

In [ ]:
record={'cap':40};audit=[]
def mock_update_cap(new_cap,role,approved):
    if role!='manager' or not approved:
        audit.append({'action':'change_cap','status':'blocked'})
        return False
    if not isinstance(new_cap,(int,float)) or new_cap<0:raise ValueError('Invalid cap')
    record['cap']=new_cap;audit.append({'action':'change_cap','status':'simulated_update'})
    return True
assert not mock_update_cap(100,'student',True) and record['cap']==40
assert mock_update_cap(45,'manager',True) and record['cap']==45
print(record,audit)
